# Module 15: Motif Discovery

**Tier 2 — Core Bioinformatics | Module 15**

*Prerequisites: Modules 1–14. Module 10 (Sequence Motifs and Domains) for PWM background.*

---

This module covers quantitative motif analysis: building Position Weight Matrices, scoring sequences, computing information content and KDIC, generating score distributions, and performing statistical enrichment testing.

**By the end of this module you will be able to:**
- Build a PPM/PWM from a set of aligned binding-site sequences
- Compute information content (IC) and KDIC to assess motif quality
- Generate an IUPAC consensus sequence
- Score genomic sequences against a PWM
- Compute score distributions (exact enumeration and Monte Carlo)
- Test motif enrichment with Fisher's exact test and BH correction

**Attribution:** *Code patterns inspired by TotipotencyLab Motif Discovery Pipeline (MIT License), Max Planck Institute of Biochemistry. Uses JASPAR 2024 public motif database.*

## Background: From Binding Sites to Matrices

Transcription factors recognize short degenerate DNA sequences (~6–20 bp). The same TF can bind many sequences because some positions tolerate multiple bases.

**Representation levels:**

| Matrix | Content | Use |
|---|---|---|
| PFM (Position Frequency Matrix) | Raw counts per base per position | Input data |
| PPM (Position Probability Matrix) | Frequencies (+ pseudocounts) | Normalized |
| PWM (Position Weight Matrix) | Log-odds vs background | Scoring |

**Information content (IC)** measures how much a position deviates from random:
- IC = 0 bits → completely degenerate (any base equally likely)
- IC = 2 bits → fully conserved (only one base allowed)
- IC per position: IC[i] = 2 + Σ_k p[i,k] · log₂(p[i,k])

**KDIC (Kullback-Discrimination Information Criterion):** mean IC across all positions, normalized to [0, 1] by dividing by 2. Useful for ranking motifs by specificity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

ALPHABET = "ACGT"
BASE_TO_IDX = {b: i for i, b in enumerate(ALPHABET)}
COLORS = {"A": "#2ecc71", "C": "#3498db", "G": "#f39c12", "T": "#e74c3c"}

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("Libraries loaded.")

## 1. Building a PPM and PWM

We start with a set of aligned sequences — typically ChIP-seq peak summits or SELEX-derived binding sites. We will use a synthetic set of CTCF-like sequences (CCGCGNGGNGGCAG consensus) for demonstration.

In [ ]:
# Synthetic CTCF-like binding sites (based on public JASPAR MA0139.1 consensus)
rng = np.random.default_rng(42)

def sample_ctcf_like(n=200):
    """Generate synthetic CTCF-like 19-mer sequences."""
    consensus_weights = [
        [0.1, 0.7, 0.1, 0.1],  # pos 0: C dominant
        [0.1, 0.7, 0.1, 0.1],  # pos 1: C dominant
        [0.1, 0.1, 0.7, 0.1],  # pos 2: G
        [0.1, 0.7, 0.1, 0.1],  # pos 3: C
        [0.1, 0.1, 0.7, 0.1],  # pos 4: G
        [0.25]*4,               # pos 5: N
        [0.1, 0.1, 0.7, 0.1],  # pos 6: G
        [0.1, 0.1, 0.7, 0.1],  # pos 7: G
        [0.25]*4,               # pos 8: N
        [0.1, 0.1, 0.7, 0.1],  # pos 9: G
        [0.1, 0.1, 0.7, 0.1],  # pos 10: G
        [0.1, 0.7, 0.1, 0.1],  # pos 11: C
        [1.0, 0.0, 0.0, 0.0],  # pos 12: A
        [0.1, 0.1, 0.7, 0.1],  # pos 13: G
    ]
    seqs = []
    for _ in range(n):
        seq = "".join(
            rng.choice(list(ALPHABET), p=w)
            for w in consensus_weights
        )
        seqs.append(seq)
    return seqs

seqs = sample_ctcf_like(200)
print(f"Generated {len(seqs)} sequences, length {len(seqs[0])}")
print("Sample:", seqs[:3])

def make_ppm(seqs, pseudocount=0.1):
    L = len(seqs[0])
    counts = np.zeros((L, 4))
    for s in seqs:
        for i, b in enumerate(s.upper()):
            if b in BASE_TO_IDX:
                counts[i, BASE_TO_IDX[b]] += 1
    ppm = (counts + pseudocount) / (len(seqs) + 4 * pseudocount)
    return ppm

def make_pwm(ppm, bg=0.25):
    return np.log2(ppm / bg)

ppm = make_ppm(seqs)
pwm = make_pwm(ppm)

print(f"\nPPM shape: {ppm.shape}  (L\u00d74: {ppm.shape[0]} positions \u00d7 4 bases)")
print(f"PPM row sums (should \u2248 1): {ppm.sum(axis=1).round(4)}")
print(f"\nPWM (first 5 positions):\n{pd.DataFrame(pwm[:5], columns=list(ALPHABET)).round(3)}")

## 2. Information Content and KDIC

IC per position quantifies how much that position constrains base identity. A fully conserved position (IC = 2 bits) contributes maximally; a degenerate position (IC ≈ 0) contributes nothing to discrimination.

**KDIC** summarizes the whole motif with one number: mean IC / 2, ranging from 0 (no information) to 1 (fully conserved). This is useful for comparing motifs of different lengths.

In [ ]:
def information_content(ppm):
    ic = 2 + np.sum(ppm * np.log2(np.clip(ppm, 1e-9, 1)), axis=1)
    ic = np.clip(ic, 0, 2)
    return ic

def kdic(ppm):
    return information_content(ppm).mean() / 2

ic = information_content(ppm)
k = kdic(ppm)

print(f"IC per position (bits): {ic.round(3)}")
print(f"Total IC: {ic.sum():.2f} bits")
print(f"KDIC: {k:.4f}  (higher = more specific motif)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# IC bar plot
axes[0].bar(range(len(ic)), ic, color=[
    "#e74c3c" if v > 1.5 else "#f39c12" if v > 0.8 else "#95a5a6"
    for v in ic
])
axes[0].axhline(1.0, color="gray", ls="--", lw=0.8, label="IC = 1 bit")
axes[0].set_xlabel("Position"); axes[0].set_ylabel("IC (bits)")
axes[0].set_title(f"Information content per position\nTotal IC = {ic.sum():.1f} bits | KDIC = {k:.3f}")
axes[0].legend()

# Sequence logo (letter heights)
logo_h = ppm * ic[:, None]  # L\u00d74 heights
x = np.arange(len(ic))
bottom = np.zeros(len(ic))
for j, base in enumerate(ALPHABET):
    axes[1].bar(x, logo_h[:, j], bottom=bottom, color=COLORS[base], label=base, width=0.8)
    bottom += logo_h[:, j]
axes[1].set_xlabel("Position"); axes[1].set_ylabel("Height (bits)")
axes[1].set_title("Sequence logo (information-content heights)")
axes[1].legend(frameon=False, ncol=4)

plt.tight_layout(); plt.show()

## 3. IUPAC Consensus Sequence

The IUPAC alphabet has degenerate codes for positions with multiple likely bases. We assign a code based on which bases exceed a frequency threshold, and mark low-IC positions as `N`.

In [ ]:
IUPAC_MAP = {
    frozenset("A"): "A", frozenset("C"): "C",
    frozenset("G"): "G", frozenset("T"): "T",
    frozenset("AG"): "R", frozenset("CT"): "Y",
    frozenset("GC"): "S", frozenset("AT"): "W",
    frozenset("GT"): "K", frozenset("AC"): "M",
    frozenset("CGT"): "B", frozenset("AGT"): "D",
    frozenset("ACT"): "H", frozenset("ACG"): "V",
    frozenset("ACGT"): "N",
}

def iupac_consensus(ppm, ic_threshold=1.0, freq_threshold=0.25):
    ic = information_content(ppm)
    cons = []
    for pos_idx, (row, ic_val) in enumerate(zip(ppm, ic)):
        if ic_val < ic_threshold:
            cons.append("N")
        else:
            dominant = frozenset(ALPHABET[j] for j, p in enumerate(row) if p >= freq_threshold)
            cons.append(IUPAC_MAP.get(dominant, "N"))
    return "".join(cons)

consensus_seq = iupac_consensus(ppm)
print(f"IUPAC consensus: {consensus_seq}")
print(f"Length: {len(consensus_seq)}")

# Compare with max-IC base at each position
max_base = "".join(ALPHABET[np.argmax(ppm[i])] for i in range(len(ppm)))
print(f"Max-probability: {max_base}")

## 4. Scoring Sequences Against a PWM

Given a PWM and a query sequence, the score is the sum of log-odds values at each position. Higher scores = better match.

For genome-wide scanning: slide the motif along every position and collect scores above a threshold. The threshold is typically set at the percentile of the score distribution.

In [ ]:
def score_sequence(seq, pwm):
    score = 0.0
    for i, b in enumerate(seq.upper()):
        j = BASE_TO_IDX.get(b, -1)
        if j >= 0 and i < len(pwm):
            score += pwm[i, j]
    return score

def scan_sequence(genome_seq, pwm, rc=True):
    """Scan a sequence and its reverse complement. Returns list of (pos, score, strand)."""
    L = len(pwm)
    hits = []
    rc_map = str.maketrans("ACGT", "TGCA")

    def _scan(seq, strand):
        for i in range(len(seq) - L + 1):
            s = score_sequence(seq[i:i+L], pwm)
            hits.append((i, s, strand))

    _scan(genome_seq, "+")
    if rc:
        _scan(genome_seq[::-1].translate(rc_map), "-")
    return hits

# Test
test_seqs = seqs[:10] + ["AAAAAAAAAAAAA" + "A" * (len(seqs[0]) - 13)]  # motif + random
scores = [score_sequence(s, pwm) for s in test_seqs]
print("Motif sequence scores:", [f"{s:.2f}" for s in scores[:5]])
print("Random sequence score:", f"{scores[-1]:.2f}")

# Score distribution visualization
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([score_sequence(s, pwm) for s in seqs], bins=30, color="steelblue", alpha=0.8, label="Motif seqs")
rnd = ["".join(rng.choice(list(ALPHABET), size=len(seqs[0]))) for _ in range(500)]
ax.hist([score_sequence(s, pwm) for s in rnd], bins=30, color="salmon", alpha=0.8, label="Random seqs")
ax.set_xlabel("PWM score"); ax.set_ylabel("Count")
ax.set_title("Score distributions: motif sites vs random sequences")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Score Distribution and Threshold Selection

To call a hit, we need a significance threshold. There are two approaches:

1. **Exact enumeration** (motif length ≤ 10): compute scores for all 4^L possible sequences, weighted by background probability
2. **Monte Carlo** (motif length > 10): sample random sequences under the background model and compute empirical p-values

The threshold is usually the score at a p-value of 10⁻⁴ to 10⁻⁶ relative to the background distribution.

In [ ]:
L = len(pwm)
if L <= 10:
    print(f"Motif length = {L} \u2264 10 \u2192 exact enumeration")
    all_scores = []
    for bases in product(range(4), repeat=L):
        s = sum(pwm[i, b] for i, b in enumerate(bases))
        all_scores.append(s)
    all_scores = np.array(all_scores)
    threshold_1e4 = np.percentile(all_scores, 99.99)  # top 0.01%
    print(f"Threshold (p < 1e-4): {threshold_1e4:.2f}")
else:
    print(f"Motif length = {L} > 10 \u2192 Monte Carlo (n=50,000)")
    mc_scores = np.array([
        score_sequence("".join(rng.choice(list(ALPHABET), size=L)), pwm)
        for _ in range(50_000)
    ])
    threshold_1e4 = np.percentile(mc_scores, 99.99)
    print(f"Threshold (p < 1e-4): {threshold_1e4:.2f}")
    all_scores = mc_scores

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_scores, bins=60, color="steelblue", alpha=0.8)
ax.axvline(threshold_1e4, color="red", lw=2, label=f"p < 1e-4 threshold ({threshold_1e4:.1f})")
ax.set_xlabel("PWM score"); ax.set_ylabel("Count")
ax.set_title(f"Background score distribution (L={L})")
ax.legend(); plt.tight_layout(); plt.show()

## 6. Motif Enrichment Testing

Given a set of peak sequences (foreground) and background sequences (e.g. GC-matched random), test whether the motif is enriched in peaks.

**Fisher's exact test:**
|          | Motif hit | No hit |
|----------|-----------|--------|
| Peaks    |     a     |   b    |
| Background |   c     |   d    |

p-value = P(observing a or more hits in peaks | H0: equal rates).

After testing multiple motifs, apply **Benjamini-Hochberg FDR** correction.

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def count_hits(sequences, pwm, threshold):
    return sum(1 for s in sequences if score_sequence(s, pwm) >= threshold)

# Simulate peak sequences (enriched for motif) and background
n_peaks = 500
n_bg    = 2000
peak_seqs = sample_ctcf_like(n_peaks)
bg_seqs   = ["".join(rng.choice(list(ALPHABET), size=len(seqs[0]))) for _ in range(n_bg)]

threshold = threshold_1e4

hits_peak = count_hits(peak_seqs, pwm, threshold)
hits_bg   = count_hits(bg_seqs,   pwm, threshold)

table = [[hits_peak,         n_peaks - hits_peak],
         [hits_bg,           n_bg    - hits_bg  ]]

_, p_value = fisher_exact(table, alternative="greater")
fold_enrich = (hits_peak / n_peaks) / max(hits_bg / n_bg, 1e-9)

print(f"Hits in peaks:      {hits_peak}/{n_peaks} ({hits_peak/n_peaks:.1%})")
print(f"Hits in background: {hits_bg}/{n_bg} ({hits_bg/n_bg:.1%})")
print(f"Fold enrichment:    {fold_enrich:.1f}\u00d7")
print(f"Fisher p-value:     {p_value:.2e}")

# Multiple motif scenario: BH correction
pvals = [p_value, 0.04, 0.12, 0.001, 0.8]  # simulated p-values for 5 motifs
_, qvals, _, _ = multipletests(pvals, method="fdr_bh")
df = pd.DataFrame({"motif": [f"motif_{i+1}" for i in range(len(pvals))],
                   "p_value": pvals, "FDR_q": qvals.round(4)})
print("\nMultiple motif FDR correction:")
print(df.to_string(index=False))

## 7. TomTom Matching (Concept)

After discovering a motif, the natural question is: *does it match a known TF?*

**TomTom** (MEME Suite) compares a query PWM against a database (JASPAR 2024, HOCOMOCO) and reports the best-matching known motifs by Euclidean distance or Pearson correlation of PWM columns.

**Key databases:**
- [JASPAR 2024](https://jaspar.elixir.no/) — curated vertebrate TF motifs, 1,210 profiles
- [HOCOMOCO v12](https://hocomoco12.autosome.ru/) — human/mouse TF motifs with quality tiers

**In Python (conceptual pattern):**
```python
# pip install pyjaspar
from pyjaspar import jaspardb
jdb = jaspardb(release="JASPAR2024")
motif = jdb.fetch_motif_by_id("MA0139.1")  # CTCF
print(motif.name, motif.ic())
```

**KDIC comparison:** sort candidate motifs by KDIC × enrichment_fold; higher-ranked motifs have both specificity and biological relevance.

In [ ]:
# Summary of key motif statistics
summary = pd.DataFrame({
    "Metric": ["Motif length (L)", "Total IC (bits)", "KDIC", "Consensus", "Threshold (p<1e-4)",
               "Peak hit rate", "BG hit rate", "Fold enrichment", "Fisher p"],
    "Value": [str(L), f"{ic.sum():.2f}", f"{k:.4f}", consensus_seq,
              f"{threshold:.2f}", f"{hits_peak/n_peaks:.1%}",
              f"{hits_bg/n_bg:.1%}", f"{fold_enrich:.1f}\u00d7", f"{p_value:.2e}"],
})
print(summary.to_string(index=False))

In [ ]:
# Exercise 15
# 1. Build a PPM from a 10-mer motif of your choice (e.g. TATAAAATAT for TATA box).
#    Compute IC and KDIC. Is it more or less specific than the CTCF motif?
# 2. Scan a 1000 bp synthetic sequence for motif hits. How does the threshold affect
#    the number of hits vs false-positive rate?
# 3. Vary the pseudocount from 0.001 to 1.0. Plot KDIC vs pseudocount.
#    What happens at very small and very large pseudocounts?
# 4. Challenge: implement a Monte Carlo p-value for a score S:
#    p = fraction of random sequences scoring >= S. Use n=10,000 samples.
#    Compare to the exact threshold for L <= 8.